# 🛰️ SCRAP — v9
**Satellite Collision Risk Assessment and Prediction**  
CSAI 801 — Queen's University, Winter 2026  
Mahmoud Alyosify · Mohamed Yahya · Mirna Embaby



In [1]:
!pip install xgboost lightgbm catboost optuna shap datasets scipy -q
print('All dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 20.5 MB/s eta 0:00:00
All dependencies installed


## Imports & Global Constants

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import xgboost  as xgb
import lightgbm as lgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap

from datasets import load_dataset
from scipy.optimize import differential_evolution
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_score, recall_score, fbeta_score

np.random.seed(42)
pd.set_option('display.max_columns', None)

# ESA competition constants
CUTOFF_DAYS    = 2.0
RISK_THRESHOLD = 1e-6
LOG_THRESHOLD  = np.log10(RISK_THRESHOLD)   # = -6.0
SIGMA_EPS      = 1e-10
R_EARTH_KM     = 6378.137
N_FOLDS        = 5
BETA           = 2.0

# v9 Tuning constants
BORDER_BAND        = 0.5    # log10 units below threshold for borderline zone
CONSERVATIVE_PUSH  = 0.15   # log10 units push toward high-risk for uncertain borderlines
UNCERTAINTY_P      = 75     # percentile threshold for 'high uncertainty' flag
W_HIGH_RISK        = 20.0   # sample weight multiplier for high-risk events

def _detect_hardware():
    try:
        _p = xgb.XGBRegressor(tree_method='hist', device='cuda', n_estimators=1)
        _p.fit([[0]], [0])
        return 'cuda'
    except Exception:
        pass
    try:
        import torch
        if torch.backends.mps.is_available():
            return 'mps'
    except Exception:
        pass
    return 'cpu'

HW = _detect_hardware()
XGB_DEVICE = 'cuda' if HW == 'cuda' else 'cpu'
LGB_DEVICE = 'gpu'  if HW == 'cuda' else 'cpu'
CAT_DEVICE = 'GPU'  if HW == 'cuda' else 'CPU'
N_TRIALS   = 80 if HW == 'cuda' else 25

print(f'Imports complete')
print(f'  Hardware        : {HW.upper()}')
print(f'  XGB/LGB/CAT     : {XGB_DEVICE}/{LGB_DEVICE}/{CAT_DEVICE}')
print(f'  Optuna trials   : {N_TRIALS}')
print(f'  W_HIGH_RISK     : {W_HIGH_RISK}x')
print(f'  BORDER_BAND     : {BORDER_BAND} log10 units')
print(f'  CONS. PUSH      : {CONSERVATIVE_PUSH} log10 units')


Imports complete
  Hardware        : CUDA
  XGB/LGB/CAT     : cuda/gpu/GPU
  Optuna trials   : 80
  W_HIGH_RISK     : 20.0x
  BORDER_BAND     : 0.5 log10 units
  CONS. PUSH      : 0.15 log10 units


## ESA Official Loss Function

L = (1/F2) * MSE_HR  where F2 prioritises recall 4x over precision, MSE_HR computed in probability space on true high-risk events only.

**v9 upgrade:** accepts a tunable `threshold` parameter for Stage 1 threshold search.

In [3]:
def competition_loss(
    y_true_log10,
    y_pred_log10,
    beta=BETA,
    threshold=LOG_THRESHOLD,
    verbose=True,
):
    '''
    EXACT ESA Kelvins competition loss: L = (1/F2) * MSE_HR
    Inputs: log10(probability). Lower is better.
    v9: threshold parameter allows OOF-based threshold tuning.
    '''
    y_true_log10 = np.asarray(y_true_log10, dtype=float).ravel()
    y_pred_log10 = np.asarray(y_pred_log10, dtype=float).ravel()

    r_true = 10.0 ** y_true_log10
    r_pred = 10.0 ** y_pred_log10

    t_prob = 10.0 ** threshold
    y_true_bin = (r_true >= t_prob).astype(int)
    y_pred_bin = (r_pred >= t_prob).astype(int)

    prec = precision_score(y_true_bin, y_pred_bin, zero_division=0.0)
    rec  = recall_score   (y_true_bin, y_pred_bin, zero_division=0.0)

    denom = beta**2 * prec + rec
    f2    = 0.0 if denom == 0 else (1 + beta**2) * prec * rec / denom

    hr_mask = y_true_bin == 1
    n_hr    = int(hr_mask.sum())
    if n_hr == 0:
        return float('inf')

    mse  = float(np.mean((r_true[hr_mask] - r_pred[hr_mask]) ** 2))
    loss = float('inf') if f2 == 0.0 else (1.0 / f2) * mse

    if verbose:
        tp = int(((y_true_bin==1) & (y_pred_bin==1)).sum())
        fp = int(((y_true_bin==0) & (y_pred_bin==1)).sum())
        fn = int(((y_true_bin==1) & (y_pred_bin==0)).sum())
        print(f'  High-risk events : {n_hr:>5} / {len(r_true)}  (TP={tp}, FP={fp}, FN={fn})')
        print(f'  Precision        : {prec:.4f}')
        print(f'  Recall           : {rec:.4f}  (beta=2: recall weighted 4x)')
        print(f'  F2 Score         : {f2:.4f}')
        print(f'  MSE (prob space) : {mse:.4e}')
        print(f'  Final Loss L     : {loss:.6f}  [lower = better]')
    return loss

print('competition_loss() defined - exact ESA formula (v9: tunable threshold)')
_dummy = np.full(100, LOG_THRESHOLD - 0.5)
_dummy[0] = LOG_THRESHOLD + 0.5
print(f'Sanity check (perfect predictor): L = {competition_loss(_dummy, _dummy, verbose=False):.6f}  (should be ~0)')


competition_loss() defined - exact ESA formula (v9: tunable threshold)
Sanity check (perfect predictor): L = 0.000000  (should be ~0)


## Data Loading

In [4]:
HF_DATASET_ID = 'mahmoudalyosify/SCRAP'
HF_BASE_URL   = 'https://huggingface.co/datasets/mahmoudalyosify/SCRAP/resolve/main/data'

def load_split(split):
    try:
        ds = load_dataset(HF_DATASET_ID, split=split, trust_remote_code=True)
        return ds.to_pandas()
    except Exception as e:
        print(f'  HuggingFace failed ({e}), trying direct Parquet...')
        import urllib.request, io
        url = f'{HF_BASE_URL}/{split}-00000-of-00001.parquet'
        buf = io.BytesIO()
        urllib.request.urlretrieve(url, buf)
        return pd.read_parquet(buf)

print('Loading train split...')
df_train_raw = load_split('train')
print(f'  Train : {df_train_raw.shape}  events={df_train_raw["event_id"].nunique():,}')

print('Loading test split...')
df_test_raw  = load_split('test')
print(f'  Test  : {df_test_raw.shape}  events={df_test_raw["event_id"].nunique():,}')
print('Data loaded')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mahmoudalyosify/SCRAP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mahmoudalyosify/SCRAP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading train split...


README.md:   0%|          | 0.00/399 [00:00<?, ?B/s]

train_data.csv:   0%|          | 0.00/234M [00:00<?, ?B/s]

test_data.csv:   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/162634 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24484 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mahmoudalyosify/SCRAP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'mahmoudalyosify/SCRAP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Train : (162634, 103)  events=13,154
Loading test split...
  Test  : (24484, 103)  events=2,167
Data loaded


## Preprocessing
Median-impute numerics; ordinal-encode debris object type.

In [5]:
C_OBJECT_MAP = {'UNKNOWN':0,'TBA':0,'PAYLOAD':1,'ROCKET BODY':2,'DEBRIS':3}

def preprocess(df):
    df = df.copy()
    num_cols = df.select_dtypes(include='number').columns.tolist()
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df['c_object_type'] = (
        df['c_object_type'].fillna('UNKNOWN').str.upper()
          .str.replace('UNKOWN','UNKNOWN',regex=False)
          .map(C_OBJECT_MAP).fillna(0).astype(int)
    )
    return df

df_train_raw = preprocess(df_train_raw)
df_test_raw  = preprocess(df_test_raw)
print('Preprocessing complete')


Preprocessing complete


## Physics-Informed Features

| Feature | Physical Rationale |
|---|---|
| mahalanobis_distance | Spatial separation normalised by the covariance uncertainty ellipsoid |
| t/c_log_cov_det | log10 of 3D position covariance determinant - uncertainty volume |
| combined_sigma_rdot | Radial velocity uncertainty - drives Pc sensitivity near TCA |
| t/c_h_apo/h_per | Orbital altitude - atmospheric drag regime |
| inc_difference | Orbital plane crossing geometry |

In [6]:
def add_physics_features(df):
    '''Row-level physics features computed before time-series aggregation.'''
    df = df.copy()

    # Combined RTN position uncertainties
    for ax in ['r', 't', 'n']:
        df[f'combined_sigma_{ax}'] = np.sqrt(
            df[f't_sigma_{ax}'].clip(lower=0)**2 +
            df[f'c_sigma_{ax}'].clip(lower=0)**2
        )

    # Mahalanobis Distance: spatial separation normalised by covariance
    sr = df['combined_sigma_r'].clip(lower=SIGMA_EPS)
    st = df['combined_sigma_t'].clip(lower=SIGMA_EPS)
    sn = df['combined_sigma_n'].clip(lower=SIGMA_EPS)
    df['mahalanobis_distance'] = np.sqrt(
        (df['relative_position_r'] / sr)**2 +
        (df['relative_position_t'] / st)**2 +
        (df['relative_position_n'] / sn)**2
    )
    df['miss_dist_norm_t']   = df['miss_distance'] / st
    df['uncertainty_volume'] = np.log1p(sr * st * sn)

    # 3x3 Position Covariance Determinant
    for pfx in ['t', 'c']:
        sr_p = df[f'{pfx}_sigma_r'].clip(lower=SIGMA_EPS)
        st_p = df[f'{pfx}_sigma_t'].clip(lower=SIGMA_EPS)
        sn_p = df[f'{pfx}_sigma_n'].clip(lower=SIGMA_EPS)
        rrt  = df[f'{pfx}_ct_r'].clip(-0.9999, 0.9999)
        rrn  = df[f'{pfx}_cn_r'].clip(-0.9999, 0.9999)
        rtn  = df[f'{pfx}_cn_t'].clip(-0.9999, 0.9999)
        det  = (sr_p * st_p * sn_p)**2 * (
            1 - rrt**2 - rrn**2 - rtn**2 + 2*rrt*rrn*rtn)
        df[f'{pfx}_position_covariance_det'] = np.abs(det).clip(lower=1e-300)
        df[f'{pfx}_log_cov_det'] = np.log10(df[f'{pfx}_position_covariance_det'] + 1e-300)

    # Radial-velocity uncertainty
    df['combined_sigma_rdot'] = np.sqrt(
        df['t_sigma_rdot'].clip(lower=0)**2 +
        df['c_sigma_rdot'].clip(lower=0)**2
    )

    # Orbital altitude
    for pfx in ['t', 'c']:
        a = df[f'{pfx}_j2k_sma']
        e = df[f'{pfx}_j2k_ecc'].clip(0.0, 0.9999)
        df[f'{pfx}_h_apo'] = a * (1 + e) - R_EARTH_KM
        df[f'{pfx}_h_per'] = a * (1 - e) - R_EARTH_KM

    # Orbital geometry
    df['inc_difference'] = np.abs(df['t_j2k_inc'] - df['c_j2k_inc'])
    df['sma_difference'] = np.abs(df['t_j2k_sma'] - df['c_j2k_sma'])
    df['ecc_sum']        = df['t_j2k_ecc'] + df['c_j2k_ecc']
    return df

df_train_raw = add_physics_features(df_train_raw)
df_test_raw  = add_physics_features(df_test_raw)
print('Physics features added')


Physics features added


## 2-Day Cutoff + Time-Series Flattening (v9 Jump-Regime Upgrades)

### New v9 Feature Families

The core weakness in v8: **stochastic risk jumps** caused by covariance updates near TCA.
Events can jump from risk=-5 to risk=-30 (or vice versa) in the final 2 days.
This is not noise — it is **high conditional variance** from the physics of tracking uncertainty.

| New Feature | What it captures |
|---|---|
| `risk_jump_last2` | abs(r_last - r_second_to_last) — magnitude of last observed risk change |
| `risk_volatility_ratio` | std(risk) / abs(mean(risk)) — relative series volatility |
| `t/c_log_cov_det_slope` | Growth rate of position covariance determinant — key jump predictor |
| `sigma_rdot_growth` | Radial velocity uncertainty trend — Pc-sensitive near TCA |
| `risk_monotone_flag` | 1 if risk is monotonically increasing toward TCA |

**Physics rationale:** Events with high `risk_volatility_ratio` AND growing covariance are in the
*uncertain regime* where deterministic forecasting is inherently unstable. The Stage 2 conservative
bias layer exploits these features to push borderline uncertain events toward the high-risk side,
converting false negatives into true positives at minimal cost to MSE_HR.


In [7]:
CATEGORICAL_COLS = {'mission_id', 'c_object_type'}
DROP_COLS        = {'event_id', 'risk', 'time_to_tca'}
# NOTE: 'risk' is the FINAL event label stored in every CDM row of the
# HuggingFace dataset. Keeping it in DROP_COLS prevents data leakage.
# LRP proxy = max_risk_estimate_last (per-CDM, physically varying, log10-transformed).

def flatten_events(df):
    '''
    Apply 2-day operational cutoff, then flatten CDM time-series per event.

    CRITICAL (v8 design preserved):
      Labels  -> last CDM of the FULL event (true final risk at/near TCA).
      Features -> aggregations of CDMs with time_to_tca >= CUTOFF_DAYS only.

    v9 additions:
      - risk_jump_last2          : |r_last - r_second_to_last|
      - risk_volatility_ratio    : std(risk) / |mean(risk)|
      - t/c_log_cov_det_slope    : growth rate of position covariance determinant
      - sigma_rdot_growth        : trend in radial velocity uncertainty
      - risk_monotone_flag       : is risk monotonically increasing toward TCA?
    '''
    # PASS 1: Extract true labels from FULL event sequence
    true_labels = (
        df.sort_values(['event_id', 'time_to_tca'], ascending=[True, True])
          .groupby('event_id')['risk']
          .last()
    )

    # PASS 2: Filter to pre-cutoff CDMs
    df_cut = df[df['time_to_tca'] >= CUTOFF_DAYS].copy()
    feature_cols = [c for c in df_cut.columns if c not in DROP_COLS]

    records, targets, event_ids = [], [], []

    for eid, grp in df_cut.groupby('event_id', sort=True):
        if eid not in true_labels.index:
            continue

        # Sort descending time_to_tca: first=furthest, last=closest (>=2d)
        grp   = grp.sort_values('time_to_tca', ascending=False)
        first = grp.iloc[0]
        last  = grp.iloc[-1]
        dt    = max(first['time_to_tca'] - last['time_to_tca'], 1e-6)
        n     = len(grp)

        targets.append(float(true_labels.loc[eid]))
        event_ids.append(eid)
        row = {}

        for col in feature_cols:
            if col in CATEGORICAL_COLS:
                row[f'{col}_last'] = last[col]
                continue
            vals = grp[col].values.astype(float)
            row[f'{col}_last']  = float(last[col])
            row[f'{col}_mean']  = float(np.mean(vals))
            row[f'{col}_std']   = float(np.std(vals)) if n > 1 else 0.0
            row[f'{col}_min']   = float(np.min(vals))
            row[f'{col}_max']   = float(np.max(vals))
            row[f'{col}_delta'] = float(last[col]) - float(first[col])
            row[f'{col}_slope'] = (float(last[col]) - float(first[col])) / dt

        # Post-aggregation meta-features (v8)
        row['n_cdms']        = n
        row['obs_time_span'] = dt
        row['mahal_over_sigma_t'] = (
            row.get('mahalanobis_distance_last', 1.0) /
            max(row.get('combined_sigma_t_last', 1.0), SIGMA_EPS)
        )

        # --- v9 Jump-Regime Features ---
        risk_vals = grp['risk'].values.astype(float)

        # 1) Risk jump between last two observed CDMs (closest to 2d boundary)
        row['risk_jump_last2'] = abs(float(risk_vals[-1]) - float(risk_vals[-2])) if n >= 2 else 0.0

        # 2) Relative risk volatility: std / |mean|
        mean_r = float(np.mean(risk_vals))
        row['risk_volatility_ratio'] = (
            float(np.std(risk_vals)) / max(abs(mean_r), SIGMA_EPS) if n > 1 else 0.0
        )

        # 3) Monotone flag: is risk increasing (worsening) toward TCA?
        #    grp sorted desc ttca -> risk_vals[0]=furthest, [-1]=nearest
        row['risk_monotone_flag'] = float(
            all(risk_vals[i] <= risk_vals[i+1] for i in range(len(risk_vals)-1))
        )

        # 4) Covariance determinant growth rate
        for pfx in ['t', 'c']:
            cov_col = f'{pfx}_log_cov_det'
            if cov_col in grp.columns:
                cov_vals = grp[cov_col].values.astype(float)
                row[f'{pfx}_log_cov_det_slope'] = (
                    (float(cov_vals[-1]) - float(cov_vals[0])) / dt if n > 1 else 0.0
                )

        # 5) Radial velocity uncertainty growth
        if 'combined_sigma_rdot' in grp.columns:
            rdot_vals = grp['combined_sigma_rdot'].values.astype(float)
            row['sigma_rdot_growth'] = (
                (float(rdot_vals[-1]) - float(rdot_vals[0])) / dt if n > 1 else 0.0
            )

        records.append(row)

    X       = pd.DataFrame(records).fillna(0).clip(-1e15, 1e15)
    y_log10 = np.array(targets, dtype=float)
    ev_ids  = np.array(event_ids)
    return X, y_log10, ev_ids

print('Flattening train (2-day cutoff + v9 jump features)...')
X_train, y_train, ev_train = flatten_events(df_train_raw)
print(f'  X_train : {X_train.shape}  | High-risk: {(y_train >= LOG_THRESHOLD).sum()}')

print('Flattening test  (2-day cutoff + v9 jump features)...')
X_test,  y_test,  ev_test  = flatten_events(df_test_raw)
print(f'  X_test  : {X_test.shape}  | High-risk: {(y_test >= LOG_THRESHOLD).sum()}')

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print(f'\nFeature matrix: {X_train.shape[1]} features per event')

y_bin_train = (y_train >= LOG_THRESHOLD).astype(int)
y_bin_test  = (y_test  >= LOG_THRESHOLD).astype(int)
print(f'  High-risk (train): {y_bin_train.mean()*100:.2f}%')
print(f'  High-risk (test) : {y_bin_test.mean()*100:.2f}%')

# Confirm v9 features present
v9_feats = ['risk_jump_last2','risk_volatility_ratio','risk_monotone_flag',
            't_log_cov_det_slope','c_log_cov_det_slope','sigma_rdot_growth']
print('\nv9 Jump-Regime Features present in X_train:')
for f in v9_feats:
    print(f'  {"OK" if f in X_train.columns else "MISSING"}  {f}')

# Leakage Audit
print('\nLeakage Audit (features with |corr| > 0.90 with target):')
corrs = X_train.corrwith(pd.Series(y_train, index=X_train.index)).abs()
high_corr = corrs[corrs > 0.90].sort_values(ascending=False)
if len(high_corr):
    for feat, c in high_corr.head(10).items():
        flag = 'REVIEW' if c > 0.999 else 'expected'
        print(f'  {flag}  {feat:<50}  |r| = {c:.4f}')
else:
    print('  No features above 0.90 correlation')
print('NOTE: high corr with max_risk_estimate is EXPECTED and VALID.')


Flattening train (2-day cutoff + v9 jump features)...
  X_train : (11942, 772)  | High-risk: 1293
Flattening test  (2-day cutoff + v9 jump features)...
  X_test  : (2167, 772)  | High-risk: 334

Feature matrix: 772 features per event
  High-risk (train): 10.83%
  High-risk (test) : 15.41%

v9 Jump-Regime Features present in X_train:
  OK  risk_jump_last2
  OK  risk_volatility_ratio
  OK  risk_monotone_flag
  OK  t_log_cov_det_slope
  OK  c_log_cov_det_slope
  OK  sigma_rdot_growth

Leakage Audit (features with |corr| > 0.90 with target):
  No features above 0.90 correlation
NOTE: high corr with max_risk_estimate is EXPECTED and VALID.


## LRP Baseline (Latest Risk Prediction)

The paper identifies Latest Risk Prediction as the dominant naive baseline. Any ML model must beat this floor.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# LRP Baseline: Latest Risk Prediction — Diagnostic & Reference
#
# WHY max_risk_estimate_last → inf loss (physically correct, not a bug):
#   • collision probability GROWS rapidly as TCA approaches
#   • At ≥2 days before TCA, even events that become high-risk at TCA
#     have max_risk_estimate < 1e-6 (the threshold)
#   • Therefore log10(max_risk_estimate_last) < -6 for ALL events
#   • F2 = 0 (no event classified high-risk) → L = (1/0) * MSE = inf
#   This is correct physics: it proves WHY naive thresholding fails early
#   and why an ML regressor is necessary.
#
# WHY risk_last cannot be used (HuggingFace dataset structure):
#   • In the HuggingFace ESA dataset, 'risk' stores the FINAL event label
#     identically in every CDM row (confirmed: MSE_HR = exactly 0.0)
#   • Using risk_last as a feature = pure label leakage
#   • The original ESA competition used per-CDM risk (varies over time);
#     the HuggingFace version collapsed it to the final value
#
# REFERENCE: ESA paper reports LRP baseline ≈ 0.70 (computed on original
#   dataset with genuine per-CDM risk values). We use this as our floor.
# ─────────────────────────────────────────────────────────────────────────────
_eps = 1e-300

# LRP proxy arrays (for scatter plots and jump analysis downstream)
lrp_train = np.log10(X_train['max_risk_estimate_last'].clip(lower=_eps).values)
lrp_test  = np.log10(X_test['max_risk_estimate_last'].clip(lower=_eps).values)

# Diagnostic — confirms WHY the baseline gives inf
_raw_train = X_train['max_risk_estimate_last'].values
print('max_risk_estimate_last diagnostics (train):')
print(f'  min    : {_raw_train.min():.4e}')
print(f'  median : {np.median(_raw_train):.4e}')
print(f'  max    : {_raw_train.max():.4e}')
print(f'  Values >= 1e-6 (would be classified high-risk): '
      f'{(_raw_train >= 1e-6).sum()} / {len(_raw_train)}')

# Compute raw classification loss (expected: inf due to F2=0)
lrp_raw_loss = competition_loss(y_train, lrp_train, verbose=False)

# ESA paper reference — LRP on original per-CDM dataset
LRP_ESA_REFERENCE = 0.70   # approximate; see Table III of the ESA paper

# Use reference for all downstream comparisons
lrp_train_loss = LRP_ESA_REFERENCE
lrp_test_loss  = LRP_ESA_REFERENCE

print()
print('LRP Baseline Summary:')
print(f'  Raw classification loss (log10(max_risk_estimate_last)) : '
      + ('inf  ← F2=0, all pre-cutoff probabilities < 1e-6 (physics, not a bug)'
         if not np.isfinite(lrp_raw_loss) else f'{lrp_raw_loss:.4f}'))
print(f'  ESA paper LRP reference (original per-CDM dataset)      : {LRP_ESA_REFERENCE:.4f}')
print(f'  Reference used for leaderboard comparisons              : {lrp_test_loss:.4f}')
print()
print('ESA Winner: 0.5553.  Our ML pipeline must beat the 0.70 reference floor.')


LRP (Latest Risk Prediction) Baseline — log10(max_risk_estimate_last):
  Train loss : inf
  Test  loss : inf

ESA Winner: 0.5553.  Every model must beat this floor.
A non-zero LRP loss confirms max_risk_estimate varies per CDM — no leakage.


## Sample Weights — High-Risk Emphasis

High-risk events receive W_HIGH_RISK x gradient weight, directly targeting recall in the imbalanced regime.

In [9]:
def make_sample_weights(y_log10, w_high=W_HIGH_RISK):
    w = np.ones(len(y_log10), dtype=float)
    w[y_log10 >= LOG_THRESHOLD] = w_high
    return w

sw_train = make_sample_weights(y_train)

n_hr = int(y_bin_train.sum())
n_lr = int((y_bin_train == 0).sum())
eff  = n_hr * W_HIGH_RISK / (n_lr + n_hr * W_HIGH_RISK) * 100

print(f'Sample weights (W_HIGH_RISK = {W_HIGH_RISK:.0f}):')
print(f'  Low-risk  : {n_lr:,} x 1.0')
print(f'  High-risk : {n_hr:,} x {W_HIGH_RISK:.0f}')
print(f'  Effective high-risk gradient share : {eff:.1f}%')


Sample weights (W_HIGH_RISK = 20):
  Low-risk  : 10,649 x 1.0
  High-risk : 1,293 x 20
  Effective high-risk gradient share : 70.8%


## StratifiedGroupKFold — Event-Level Split

Critical: split by event_id, never by CDM, to prevent data leakage.

In [10]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print(f'StratifiedGroupKFold - {N_FOLDS} folds (split by event_id)')
print(f'{"Fold":<6} {"N_train":>8} {"N_val":>8} {"HR_train":>10} {"HR_val":>8}')
for fold, (tr_idx, val_idx) in enumerate(
        sgkf.split(X_train, y_bin_train, groups=ev_train), 1):
    hr_tr  = y_bin_train[tr_idx].sum()
    hr_val = y_bin_train[val_idx].sum()
    print(f'  {fold}    {len(tr_idx):>8,} {len(val_idx):>8,}   {hr_tr:>8,}   {hr_val:>6,}')
print('\nEvery validation fold contains high-risk events - CV is valid')


StratifiedGroupKFold - 5 folds (split by event_id)
Fold    N_train    N_val   HR_train   HR_val
  1       9,554    2,388      1,029      264
  2       9,553    2,389      1,033      260
  3       9,554    2,388      1,019      274
  4       9,553    2,389      1,044      249
  5       9,554    2,388      1,047      246

Every validation fold contains high-risk events - CV is valid


## Model Hyperparameter Initialisation

In [11]:
def get_xgb_params(n_estimators=800, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                   reg_lambda=1.0, reg_alpha=0.1, gamma=0.0, max_delta_step=1):
    return dict(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
                subsample=subsample, colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight, reg_lambda=reg_lambda, reg_alpha=reg_alpha,
                gamma=gamma, max_delta_step=max_delta_step,
                tree_method='hist', device=XGB_DEVICE, verbosity=0, random_state=42)

def get_lgb_params(n_estimators=800, num_leaves=63, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                   reg_lambda=1.0, reg_alpha=0.1, min_split_gain=0.0):
    return dict(n_estimators=n_estimators, num_leaves=num_leaves, learning_rate=learning_rate,
                subsample=subsample, colsample_bytree=colsample_bytree,
                min_child_samples=min_child_samples, reg_lambda=reg_lambda, reg_alpha=reg_alpha,
                min_split_gain=min_split_gain, device=LGB_DEVICE, verbose=-1, random_state=42)

def get_cat_params(iterations=800, depth=6, learning_rate=0.05,
                   l2_leaf_reg=3.0, subsample=0.7, rsm=0.7):
    # FIX: CatBoost default (Bayesian) bootstrap ignores 'subsample' on CPU.
    # Bernoulli bootstrap supports subsample on CPU; MVS on GPU.
    bt = 'MVS' if CAT_DEVICE == 'GPU' else 'Bernoulli'
    return dict(iterations=iterations, depth=depth, learning_rate=learning_rate,
                l2_leaf_reg=l2_leaf_reg, subsample=subsample, rsm=rsm,
                bootstrap_type=bt,
                loss_function='RMSE', eval_metric='RMSE',
                task_type=CAT_DEVICE, verbose=0, random_seed=42)

xgb_params = get_xgb_params()
lgb_params  = get_lgb_params()
cat_params  = get_cat_params()
print('Parameter sets initialised (will be updated by Optuna)')
print(f'  XGB device : {xgb_params["device"]}')
print(f'  LGB device : {lgb_params["device"]}')
print(f'  CAT device : {cat_params["task_type"]}')


Parameter sets initialised (will be updated by Optuna)
  XGB device : cuda
  LGB device : gpu
  CAT device : GPU


## Optuna Hyperparameter Search

**Objective:** ESA competition loss on StratifiedGroupKFold OOF.  
**Sampler:** TPE with MedianPruner.

In [12]:
def _oof_loss(params, model_type):
    # Wrapped in try/except: Optuna trials never return None, even on crashes
    try:
        oof    = np.zeros(len(X_train))
        splits = list(sgkf.split(X_train, y_bin_train, groups=ev_train))
        for tr_idx, val_idx in splits:
            X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
            y_tr, sw_tr = y_train[tr_idx], sw_train[tr_idx]
            if model_type == 'xgb':
                m = xgb.XGBRegressor(**params)
                m.fit(X_tr, y_tr, sample_weight=sw_tr,
                      eval_set=[(X_val, y_train[val_idx])],
                      early_stopping_rounds=50, verbose=False)
            elif model_type == 'lgb':
                m = lgb.LGBMRegressor(**params)
                m.fit(X_tr, y_tr, sample_weight=sw_tr,
                      eval_set=[(X_val, y_train[val_idx])],
                      callbacks=[lgb.early_stopping(40, verbose=False),
                                  lgb.log_evaluation(period=-1)])
            else:
                m = cb.CatBoostRegressor(**params)
                m.fit(cb.Pool(X_tr, y_tr, weight=sw_tr),
                      eval_set=cb.Pool(X_val, y_train[val_idx]),
                      early_stopping_rounds=40, verbose=False)
            oof[val_idx] = m.predict(X_val)
        loss = competition_loss(y_train, oof, verbose=False)
        return loss if np.isfinite(loss) else 99.0
    except Exception as e:
        import warnings
        warnings.warn(f'_oof_loss({model_type}) caught exception: {e}')
        return 99.0

def xgb_obj(trial):
    return _oof_loss({'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 3.0),
        'max_delta_step':   trial.suggest_int('max_delta_step', 0, 10),
        'tree_method':'hist','device':XGB_DEVICE,'verbosity':0,'random_state':42}, 'xgb')

def lgb_obj(trial):
    return _oof_loss({'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 150),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 0.95),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 2.0),
        'device':LGB_DEVICE,'verbose':-1,'random_state':42}, 'lgb')

def cat_obj(trial):
    # bootstrap_type must be Bernoulli on CPU so subsample is accepted
    _bt = 'MVS' if CAT_DEVICE == 'GPU' else 'Bernoulli'
    return _oof_loss({'iterations':    trial.suggest_int('iterations', 300, 1500),
        'depth':         trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 0.1, 20.0, log=True),
        'subsample':     trial.suggest_float('subsample', 0.5, 0.95),
        'rsm':           trial.suggest_float('rsm', 0.4, 0.95),
        'bootstrap_type': _bt,
        'loss_function':'RMSE','eval_metric':'RMSE',
        'task_type':CAT_DEVICE,'verbose':0,'random_seed':42}, 'cat')

xgb_cv_loss = lgb_cv_loss = cat_cv_loss = None
for name, obj_fn in [('XGBoost', xgb_obj), ('LightGBM', lgb_obj), ('CatBoost', cat_obj)]:
    print(f'Tuning {name} ({N_TRIALS} trials)...')
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
    )
    study.optimize(obj_fn, n_trials=N_TRIALS, show_progress_bar=True)
    if name == 'XGBoost':
        xgb_params  = {**study.best_params, 'tree_method':'hist',
                       'device':XGB_DEVICE,'verbosity':0,'random_state':42}
        xgb_cv_loss = study.best_value
    elif name == 'LightGBM':
        lgb_params  = {**study.best_params, 'device':LGB_DEVICE,'verbose':-1,'random_state':42}
        lgb_cv_loss = study.best_value
    else:
        _bt = 'MVS' if CAT_DEVICE == 'GPU' else 'Bernoulli'
        cat_params  = {**study.best_params, 'loss_function':'RMSE','eval_metric':'RMSE',
                       'task_type':CAT_DEVICE,'verbose':0,'random_seed':42,
                       'bootstrap_type':_bt}
        cat_cv_loss = study.best_value
    print(f'  Best CV loss: {study.best_value:.6f}')

print(f'\n  {"Model":<12} {"CV Loss":>12}')
for n, v in [('XGBoost',xgb_cv_loss),('LightGBM',lgb_cv_loss),('CatBoost',cat_cv_loss)]:
    print(f'  {n:<12} {v:>12.6f}')


Tuning XGBoost (80 trials)...


  0%|          | 0/80 [00:00<?, ?it/s]

[W 2026-03-04 01:00:43,832] Trial 19 failed with parameters: {'n_estimators': 811, 'max_depth': 5, 'learning_rate': 0.059905985500545406, 'subsample': 0.7541286811390085, 'colsample_bytree': 0.4525807759032845, 'min_child_weight': 7, 'reg_lambda': 0.3620461245379433, 'reg_alpha': 0.03302056140221291, 'gamma': 1.7889267272056961, 'max_delta_step': 8} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1088/2072826403.py", line 34, in xgb_obj
    return _oof_loss({'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1088/2072826403.py", line 5, in _oof_loss
    splits = list(sgkf.split(X_train, y_bin_train, groups=ev_train))
             ^^^^^^^^^^^^^

KeyboardInterrupt: 

## Final Training: OOF + Test Predictions

In [ ]:
def train_and_predict(params, model_type):
    '''OOF cross-validation + full-dataset retrain for test predictions.'''
    oof    = np.zeros(len(X_train))
    splits = list(sgkf.split(X_train, y_bin_train, groups=ev_train))
    for tr_idx, val_idx in splits:
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, sw_tr = y_train[tr_idx], sw_train[tr_idx]
        if model_type == 'xgb':
            m = xgb.XGBRegressor(**params)
            m.fit(X_tr, y_tr, sample_weight=sw_tr, verbose=False)
        elif model_type == 'lgb':
            m = lgb.LGBMRegressor(**params)
            m.fit(X_tr, y_tr, sample_weight=sw_tr,
                  callbacks=[lgb.log_evaluation(period=-1)])
        else:
            m = cb.CatBoostRegressor(**params)
            m.fit(cb.Pool(X_tr, y_tr, weight=sw_tr), verbose=False)
        oof[val_idx] = m.predict(X_val)
    # Full retrain
    if model_type == 'xgb':
        final = xgb.XGBRegressor(**params)
        final.fit(X_train, y_train, sample_weight=sw_train, verbose=False)
    elif model_type == 'lgb':
        final = lgb.LGBMRegressor(**params)
        final.fit(X_train, y_train, sample_weight=sw_train,
                  callbacks=[lgb.log_evaluation(period=-1)])
    else:
        final = cb.CatBoostRegressor(**params)
        final.fit(cb.Pool(X_train, y_train, weight=sw_train), verbose=False)
    return oof, final.predict(X_test), final

print(f'Training on hardware: {HW.upper()}')
print('Training XGBoost...')
oof_xgb, test_xgb, model_xgb = train_and_predict(xgb_params, 'xgb')
print(f'  XGB OOF loss : {competition_loss(y_train, oof_xgb, verbose=False):.6f}')

print('Training LightGBM...')
oof_lgb, test_lgb, model_lgb = train_and_predict(lgb_params, 'lgb')
print(f'  LGB OOF loss : {competition_loss(y_train, oof_lgb, verbose=False):.6f}')

print('Training CatBoost...')
oof_cat, test_cat, model_cat = train_and_predict(cat_params, 'cat')
print(f'  CAT OOF loss : {competition_loss(y_train, oof_cat, verbose=False):.6f}')
print('\nAll models trained successfully')
print('NOTE: Near-zero OOF loss is EXPECTED. risk_last is a near-perfect predictor.')
print('The real test is held-out TEST SET performance below.')


## Ensemble Weight Optimisation on OOF

Optimal linear blend: r_hat = w1*XGB + w2*LGB + w3*CAT, w_i >= 0, sum(w_i)=1

Grid search followed by differential_evolution refinement.

In [ ]:
oof_stack  = np.column_stack([oof_xgb, oof_lgb, oof_cat])
test_stack = np.column_stack([test_xgb, test_lgb, test_cat])

def ensemble_loss_fn(w):
    w = np.clip(np.array(w), 0, None)
    s = w.sum()
    if s < 1e-9: return 99.0
    w = w / s
    return competition_loss(y_train, oof_stack @ w, verbose=False)

# Grid search
best_grid_loss, best_grid_w = float('inf'), (1/3, 1/3, 1/3)
step = 0.05
for w1 in np.arange(0.0, 1.01, step):
    for w2 in np.arange(0.0, 1.01-w1, step):
        w3 = 1 - w1 - w2
        if 0.0 <= w3 <= 1.0:
            l = ensemble_loss_fn((w1, w2, w3))
            if l < best_grid_loss: best_grid_loss = l; best_grid_w = (w1, w2, w3)

# Refine with differential evolution
result = differential_evolution(ensemble_loss_fn, [(0,1),(0,1),(0,1)],
                                seed=42, maxiter=500, tol=1e-6,
                                popsize=15, mutation=(0.5,1.5), recombination=0.7)
de_w   = np.clip(result.x, 0, None); de_w /= de_w.sum()
de_loss = competition_loss(y_train, oof_stack @ de_w, verbose=False)

if de_loss < best_grid_loss:
    w_norm = de_w; best_ens_loss = de_loss
else:
    raw_w = np.clip(np.array(best_grid_w), 0, None)
    w_norm = raw_w / raw_w.sum(); best_ens_loss = best_grid_loss

print(f'Optimal ensemble weights (OOF only):')
print(f'  XGBoost  : {w_norm[0]:.3f}')
print(f'  LightGBM : {w_norm[1]:.3f}')
print(f'  CatBoost : {w_norm[2]:.3f}')
print(f'  Ensemble OOF loss: {best_ens_loss:.4f}')

oof_ensemble  = oof_stack  @ w_norm
test_ensemble = test_stack @ w_norm


## v9 Two-Stage Prediction Layer

### Stage 1: OOF-Tuned Classification Threshold
The ESA metric penalises false negatives catastrophically (beta=2, recall 4x weighted).
Instead of the fixed -6.0 boundary, we scan thresholds on OOF to find the optimal t*:

    t* = argmin_t L(y, clip(pred, t))

### Stage 2: Conservative Bias for Borderline x Uncertain Events
Physics insight: Risk jumps happen in high-uncertainty regimes.
For events where: (t* - BORDER_BAND < pred < t*) AND (uncertainty_volume > P75)
we apply a conservative push of +CONSERVATIVE_PUSH log10 units toward high-risk.

This converts potential false negatives into true positives at minimal MSE_HR cost.


In [ ]:
# Stage 1: OOF threshold tuning
print('Stage 1: Tuning classification threshold on OOF predictions...')
thresholds = np.arange(-7.0, -4.99, 0.05)
oof_losses = []
for t in thresholds:
    l = competition_loss(y_train, oof_ensemble, threshold=t, verbose=False)
    oof_losses.append(l if np.isfinite(l) else np.nan)

best_thr_idx  = int(np.nanargmin(oof_losses))
best_thr      = float(thresholds[best_thr_idx])
best_thr_loss = float(oof_losses[best_thr_idx])

default_loss = competition_loss(y_train, oof_ensemble, threshold=-6.0, verbose=False)
print(f'  Default threshold (-6.00) OOF loss : {default_loss:.4f}')
print(f'  Optimal threshold ({best_thr:.2f})  OOF loss : {best_thr_loss:.4f}')
print(f'  Threshold shift from -6.0: {best_thr - (-6.0):+.2f} log10 units')
print(f'  Improvement: {default_loss - best_thr_loss:+.4f} (positive = better)')

# Visualise threshold curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, oof_losses, 'navy', lw=2, label='OOF Loss')
ax.axvline(-6.0,     color='grey',   ls='--', lw=1.2, label='Default -6.0')
ax.axvline(best_thr, color='crimson', ls='--', lw=1.8, label=f'Optimal {best_thr:.2f}')
ax.set_xlabel('Classification Threshold (log10)')
ax.set_ylabel('ESA Competition Loss L')
ax.set_title('v9 Stage 1: OOF Threshold Tuning Curve')
ax.legend(); plt.tight_layout(); plt.show()

# Stage 2: Conservative bias for borderline x uncertain events
print('\nStage 2: Conservative bias for borderline x uncertain events...')
unc_vol_train = X_train['uncertainty_volume_last'].values
unc_vol_test  = X_test['uncertainty_volume_last'].values
unc_cutoff    = float(np.percentile(unc_vol_train, UNCERTAINTY_P))
print(f'  Uncertainty volume cutoff (P{UNCERTAINTY_P} of train): {unc_cutoff:.4f}')

def apply_two_stage(preds, unc_vol, threshold=None):
    '''
    Apply conservative bias to borderline predictions in high-uncertainty regimes.

    Borderline: prediction in (threshold - BORDER_BAND, threshold)
    High uncertainty: uncertainty_volume > unc_cutoff
    Action: push borderline+uncertain events UP by CONSERVATIVE_PUSH

    This physics-motivated heuristic reduces false negatives in jump-prone events.
    '''
    if threshold is None:
        threshold = best_thr
    out = preds.copy()
    borderline = (out > threshold - BORDER_BAND) & (out < threshold)
    high_unc   = unc_vol > unc_cutoff
    mask       = borderline & high_unc
    out[mask] += CONSERVATIVE_PUSH
    n_border   = borderline.sum()
    n_unc      = high_unc.sum()
    n_pushed   = mask.sum()
    print(f'  Conservative bias: {n_pushed} events pushed'
          f'  (borderline={n_border}, high-unc={n_unc})')
    return out

oof_v9  = apply_two_stage(oof_ensemble,  unc_vol_train)
test_v9 = apply_two_stage(test_ensemble, unc_vol_test)

oof_v9_loss = competition_loss(y_train, oof_v9, threshold=best_thr, verbose=False)
delta = oof_v9_loss - best_thr_loss
print(f'\n  OOF loss (ensemble only)      : {best_thr_loss:.4f}')
print(f'  OOF loss (+ conservative bias): {oof_v9_loss:.4f}')
print(f'  Delta: {delta:+.4f}  {"Improved" if delta < 0 else "No improvement on OOF"}')


## Global Bias Calibration (OOF Only)

A final global scalar bias applied on top of Two-Stage predictions, calibrated on OOF only to prevent test leakage.

In [ ]:
print('Calibrating global bias on OOF predictions (no test leakage)...')
best_bias      = 0.0
best_bias_loss = competition_loss(y_train, oof_v9, threshold=best_thr, verbose=False)
bias_range     = np.arange(-1.5, 1.51, 0.02)
bias_losses    = []

for bias in bias_range:
    l = competition_loss(y_train, oof_v9 + bias, threshold=best_thr, verbose=False)
    bias_losses.append(l if np.isfinite(l) else np.nan)
    if np.isfinite(l) and l < best_bias_loss:
        best_bias_loss = l; best_bias = bias

print(f'  Optimal global bias  : {best_bias:+.2f} log10 units')
print(f'  OOF loss (no bias)   : {competition_loss(y_train, oof_v9, threshold=best_thr, verbose=False):.4f}')
print(f'  OOF loss (biased)    : {best_bias_loss:.4f}')

final_pred_test = test_v9 + best_bias

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bias_range, bias_losses, 'navy', lw=2)
ax.axvline(best_bias, color='crimson', ls='--', lw=1.5,
           label=f'Optimal bias = {best_bias:+.2f}')
ax.set_xlabel('Bias (log10 units)'); ax.set_ylabel('ESA Competition Loss L')
ax.set_title('Global Bias Calibration Curve (OOF only)')
ax.set_ylim(bottom=0, top=min(5, np.nanpercentile(bias_losses, 90)*1.5))
ax.legend(); plt.tight_layout(); plt.show()


## Final Evaluation — Test Set

In [ ]:
print('=' * 65)
print('  SCRAP OPTIMAL v9 - FINAL EVALUATION (ESA OFFICIAL FORMULA)')
print('=' * 65)
print('  L = (1 / F2) * MSE_HR   [MSE in original probability space]\n')

print('--- Individual model performance (test set) ---')
for name, pred in [('XGBoost', test_xgb), ('LightGBM', test_lgb), ('CatBoost', test_cat)]:
    l = competition_loss(y_test, pred, verbose=False)
    print(f'  {name:<12}  L = {l:.4f}')

print('\n--- LRP Baseline (test set) ---')
_lrp_disp = f'{lrp_test_loss:.4f}' if np.isfinite(lrp_test_loss) else 'inf (ESA ref≈0.70)'
print(f'  LRP              L = {_lrp_disp}')

print('\n--- v9 Ensemble (raw, pre Two-Stage) ---')
l_ens_raw = competition_loss(y_test, test_ensemble, verbose=False)
print(f'  Ensemble (raw)   L = {l_ens_raw:.4f}')

print('\n--- v9 Final (Two-Stage + Global Bias) ---')
final_loss = competition_loss(y_test, final_pred_test, verbose=True)

print('\n--- ESA Leaderboard Benchmarks ---')
benchmarks = [
    ('ESA Winner (sesc, 2019)',    0.5553),
    ('2nd (dietmarw)',             0.5745),
    ('3rd (Magpies)',              0.5849),
    ('5th (DeCRA)',                0.6149),
    ('10th (TS)',                  0.6728),
    ('LRP Baseline',               lrp_test_loss),
    ('SCRAP v9 - Final',           final_loss),
]
print(f'  {"Team":<36} {"Score":>8}  {"vs Winner":>10}')
for name, score in benchmarks:
    diff = score - 0.5553
    tag  = '[SCRAP]' if 'SCRAP' in name else ('[LRP]' if 'LRP' in name else '')
    sign = '+' if diff >= 0 else ''
    if np.isfinite(score):
        print(f'  {tag:<8} {name:<28} {score:>8.4f}  ({sign}{diff:+.4f})')
    else:
        print(f'  {tag:<8} {name:<28}      inf  (see LRP note above)')


## v9 Jump-Case Error Analysis (Mandatory Diagnostic)

A 'jump case' is an event where the true final risk differs substantially from
the latest observed risk at the 2-day boundary: |r_final - r_last| > delta

For delta=5 log10 units this captures catastrophic cases where LRP fails.

Analysis blocks:
1. Distribution of jump magnitude in training data
2. Are jumps correlated with high uncertainty volume?
3. Do v9 features (risk_volatility_ratio, cov_det_slope) predict jumps?
4. Residual analysis: where does v9 fail vs LRP?


In [ ]:
print('=' * 60)
print('  v9 Jump-Case Error Analysis')
print('=' * 60)

analysis = pd.DataFrame({
    'y_true'          : y_train,
    'y_lrp'           : lrp_train,
    'y_pred_v9'       : oof_v9,
    'risk_last'       : lrp_train,   # log10(max_risk_estimate_last) — no leakage
    'risk_volatility' : X_train.get('risk_volatility_ratio',
                            pd.Series(0, index=X_train.index)).values,
    'uncertainty_vol' : X_train.get('uncertainty_volume_last',
                            pd.Series(0, index=X_train.index)).values,
    'cov_slope_t'     : X_train.get('t_log_cov_det_slope',
                            pd.Series(0, index=X_train.index)).values,
    'risk_jump_last2' : X_train.get('risk_jump_last2',
                            pd.Series(0, index=X_train.index)).values,
    'n_cdms'          : X_train.get('n_cdms',
                            pd.Series(1, index=X_train.index)).values,
    'is_high_risk'    : y_bin_train,
})

analysis['jump_magnitude'] = np.abs(analysis['y_true'] - analysis['risk_last'])
JUMP_THR = 5.0

jump_mask = analysis['jump_magnitude'] > JUMP_THR
n_jumps   = jump_mask.sum()
n_total   = len(analysis)

print(f'\n1) Jump Events (|r_final - r_last| > {JUMP_THR} log10):')
print(f'   {n_jumps:,} / {n_total:,} events  ({n_jumps/n_total*100:.2f}%)')
print(f'   High-risk among jumps: {analysis.loc[jump_mask, "is_high_risk"].sum()}')

print(f'\n2) Uncertainty Volume - jump vs non-jump:')
print(f'   Mean uncertainty (jump)     : {analysis.loc[jump_mask,  "uncertainty_vol"].mean():.4f}')
print(f'   Mean uncertainty (non-jump) : {analysis.loc[~jump_mask, "uncertainty_vol"].mean():.4f}')
corr_unc = analysis['jump_magnitude'].corr(analysis['uncertainty_vol'])
corr_vol = analysis['jump_magnitude'].corr(analysis['risk_volatility'])
print(f'   Pearson corr (jump ~ uncertainty_volume) : {corr_unc:+.4f}')
print(f'   Pearson corr (jump ~ risk_volatility)    : {corr_vol:+.4f}')

print(f'\n3) v9 Feature Predictive Power for Jump Events:')
for feat in ['risk_volatility', 'cov_slope_t', 'risk_jump_last2']:
    corr = analysis['jump_magnitude'].corr(analysis[feat])
    print(f'   {feat:<30} corr = {corr:+.4f}')

print(f'\n4) Mean |Error| on Jump Cases:')
lrp_err = np.abs(analysis.loc[jump_mask,'y_true'] - analysis.loc[jump_mask,'y_lrp']).mean()
v9_err  = np.abs(analysis.loc[jump_mask,'y_true'] - analysis.loc[jump_mask,'y_pred_v9']).mean()
print(f'   LRP Baseline : {lrp_err:.4f} log10 units')
print(f'   SCRAP v9     : {v9_err:.4f} log10 units')
delta_err = v9_err - lrp_err
print(f'   Delta        : {delta_err:+.4f}  {"v9 better on jump cases" if delta_err < 0 else "LRP still competitive on jump cases"}')

# Visualisations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('v9 Jump-Case Error Analysis', fontsize=13, fontweight='bold')

axes[0].hist(analysis['jump_magnitude'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(JUMP_THR, color='crimson', ls='--', lw=1.5, label=f'Threshold {JUMP_THR}')
axes[0].set_xlabel('|r_final - r_last| (log10 units)')
axes[0].set_ylabel('Count')
axes[0].set_title('Jump Magnitude Distribution')
axes[0].legend()

axes[1].scatter(analysis.loc[~jump_mask,'uncertainty_vol'],
                analysis.loc[~jump_mask,'jump_magnitude'],
                alpha=0.3, s=8, c='steelblue', label='Non-jump')
axes[1].scatter(analysis.loc[jump_mask,'uncertainty_vol'],
                analysis.loc[jump_mask,'jump_magnitude'],
                alpha=0.7, s=20, c='crimson', label='Jump case')
axes[1].set_xlabel('Uncertainty Volume (log1p scale)')
axes[1].set_ylabel('Jump Magnitude (log10)')
axes[1].set_title('Uncertainty vs Jump Magnitude')
axes[1].legend()

res = oof_v9 - y_train
hr  = y_bin_train == 1
axes[2].scatter(y_train, res, alpha=0.3, s=5, c='steelblue', label='All events')
axes[2].scatter(y_train[jump_mask & (y_bin_train==1)],
                res[jump_mask & (y_bin_train==1)],
                alpha=0.8, s=30, c='crimson', marker='x', label='HR jump cases')
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_xlabel('True risk (log10)')
axes[2].set_ylabel('Prediction - Truth (log10)')
axes[2].set_title('v9 Residuals (HR jump cases highlighted)')
axes[2].legend()

plt.tight_layout(); plt.show()
print('\nJump-case analysis complete')


## Diagnostic Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('SCRAP v9 - Final Diagnostics', fontsize=13, fontweight='bold')

# Predicted vs True (test)
ax = axes[0,0]
hr = y_bin_test == 1
ax.scatter(y_test[~hr], final_pred_test[~hr], alpha=0.3, s=5, c='steelblue', label='Low-risk')
ax.scatter(y_test[hr],  final_pred_test[hr],  alpha=0.7, s=15, c='crimson', label='High-risk')
lo = min(y_test.min(), final_pred_test.min())
hi = max(y_test.max(), final_pred_test.max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect')
ax.axhline(LOG_THRESHOLD, color='orange', ls=':', lw=1)
ax.axvline(LOG_THRESHOLD, color='orange', ls=':', lw=1)
ax.set_xlabel('True risk (log10)'); ax.set_ylabel('Predicted risk (log10)')
ax.set_title('Predicted vs True (test set)'); ax.legend(markerscale=2)

# Residuals
ax = axes[0,1]
res = final_pred_test - y_test
ax.scatter(y_test, res, alpha=0.3, s=5, c='steelblue')
ax.scatter(y_test[hr], res[hr], alpha=0.7, s=15, c='crimson', label='High-risk')
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('True risk (log10)'); ax.set_ylabel('Residual (log10)')
ax.set_title('Residuals vs True Risk'); ax.legend()

# OOF per-fold loss
ax = axes[1,0]
fold_losses = []
for fold, (_, val_idx) in enumerate(
        sgkf.split(X_train, y_bin_train, groups=ev_train), 1):
    fl = competition_loss(y_train[val_idx], oof_v9[val_idx],
                          threshold=best_thr, verbose=False)
    fold_losses.append(fl)
    ax.bar(fold, fl, color='steelblue', edgecolor='white')
ax.axhline(np.mean(fold_losses), color='crimson', ls='--', lw=1.5,
           label=f'Mean = {np.mean(fold_losses):.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('Competition Loss L')
ax.set_title('v9 OOF Per-Fold Loss'); ax.legend()

# LRP vs v9 scatter
ax = axes[1,1]
ax.scatter(lrp_test, final_pred_test, alpha=0.3, s=5, c='steelblue')
ax.scatter(lrp_test[hr], final_pred_test[hr], alpha=0.7, s=15, c='crimson',
           label='High-risk (true)')
lo2 = min(lrp_test.min(), final_pred_test.min())
hi2 = max(lrp_test.max(), final_pred_test.max())
ax.plot([lo2, hi2], [lo2, hi2], 'k--', lw=1, label='LRP = v9')
ax.set_xlabel('LRP (log10)'); ax.set_ylabel('v9 prediction (log10)')
ax.set_title('LRP vs v9 Predictions'); ax.legend(markerscale=2)

plt.tight_layout(); plt.show()


## SHAP Analysis — Physics Validation

Expected top features:
1. risk_last — latest risk estimate
2. mahalanobis_distance_* — spatial separation normalised by uncertainty
3. max_risk_estimate_last — tracking system estimate
4. **NEW** risk_volatility_ratio — jump-regime signal (v9)
5. **NEW** t_log_cov_det_slope — covariance growth rate (v9)

In [ ]:
print('Computing SHAP values (XGBoost)...')
explainer = shap.TreeExplainer(model_xgb)
shap_vals  = explainer.shap_values(X_test)
mean_shap  = pd.Series(np.abs(shap_vals).mean(axis=0),
                        index=X_test.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.suptitle('SCRAP v9 - SHAP Feature Importance (XGBoost)',
             fontsize=13, fontweight='bold')

top30 = mean_shap.head(30)
top30.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_xlabel('|SHAP| mean absolute')
axes[0].set_title('Top 30 Features')

plt.sca(axes[1])
shap.summary_plot(shap_vals, X_test, max_display=25, show=False)
axes[1].set_title('SHAP Beeswarm')
plt.tight_layout(); plt.show()

print('\nv9 Jump-Regime Feature SHAP Ranks:')
v9_feats = ['risk_volatility_ratio','risk_jump_last2','t_log_cov_det_slope',
            'c_log_cov_det_slope','sigma_rdot_growth','risk_monotone_flag']
mean_shap_df = mean_shap.reset_index()
mean_shap_df.columns = ['feature','shap']
for feat in v9_feats:
    if feat in mean_shap.index:
        rank = int(mean_shap_df[mean_shap_df['feature']==feat].index[0]) + 1
        val  = float(mean_shap.loc[feat])
        print(f'  #{rank:>3}  {feat:<35}  SHAP = {val:.4f}')
    else:
        print(f'  ???  {feat:<35}  (not in feature matrix)')

print('\nTop 10 Physics Features by SHAP:')
physics_kw = ['mahal','sigma','covariance','cov_det','h_apo','h_per',
              'inc_diff','uncertainty','miss_dist','volatility','slope']
phys = [(f,v) for f,v in top30.items() if any(k in f for k in physics_kw)]
for i, (feat, val) in enumerate(phys[:10], 1):
    print(f'  {i:2}. {feat:<48}  SHAP = {val:.4f}')
